# SGLang 模型评估

这个 notebook 用于快速评估本地 SGLang 服务上的模型效果、JSON 输出和基础延迟。

默认目标：`http://127.0.0.1:30000/v1`，模型：`qwen35-4b`。可通过 `SGLANG_BASE_URL`、`SGLANG_API_KEY`、`SGLANG_MODEL` 覆盖。

In [ ]:
import os
import time
import statistics
import requests
from openai import OpenAI

BASE_URL = os.getenv("SGLANG_BASE_URL", "http://127.0.0.1:30000/v1")
API_KEY = os.getenv("SGLANG_API_KEY", "EMPTY")
MODEL = os.getenv("SGLANG_MODEL", "qwen35-4b")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("BASE_URL:", BASE_URL)
print("MODEL   :", MODEL)
print("API_KEY :", "<set>" if API_KEY and API_KEY != "EMPTY" else API_KEY)


In [ ]:
# 1) 服务健康检查 + 模型列表
headers = {"Authorization": f"Bearer {API_KEY}"}
health_url = BASE_URL.replace("/v1", "") + "/model_info"
health = requests.get(health_url, headers=headers, timeout=10)
print("health url:", health_url)
print("/model_info status:", health.status_code)
print(health.text[:300])

models = client.models.list()
print("\n/models count:", len(models.data))
for m in models.data:
    print("-", m.id)


In [ ]:
# 2) 通用调用函数：返回文本 + 延迟 + token 统计
def chat_once(prompt, system="你是一个严谨、简洁的中文助手", temperature=0.2, max_tokens=256):
    start = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    cost = time.time() - start
    text = resp.choices[0].message.content
    usage = getattr(resp, "usage", None)
    prompt_tokens = getattr(usage, "prompt_tokens", None) if usage else None
    completion_tokens = getattr(usage, "completion_tokens", None) if usage else None
    return {
        "text": text,
        "latency_s": round(cost, 3),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }


In [ ]:
# 3) 效果样例（中文能力 + 推理 + 信息抽取）
prompts = [
    "把这句话压缩成 15 个字以内：今天团队把线上告警处理完并补充了监控。",
    "用三点解释什么是幂等设计，并给一个支付场景例子。",
    "从文本中抽取 JSON：订单A 金额120.5元 状态已支付 用户张三",
]

for i, p in enumerate(prompts, 1):
    out = chat_once(p, temperature=0.1, max_tokens=220)
    print("\n=== Case", i, "===")
    print("Prompt:", p)
    print("Latency(s):", out["latency_s"])
    print("Tokens:", out["prompt_tokens"], out["completion_tokens"])
    print("Answer:", out["text"])


In [ ]:
# 4) 简单延迟稳定性测试（同一提示词多次调用）
benchmark_prompt = "请用 80 字介绍一下任务路由系统的核心价值。"
rounds = 8
latencies = []

for _ in range(rounds):
    out = chat_once(benchmark_prompt, temperature=0.0, max_tokens=160)
    latencies.append(out["latency_s"])

print("latencies:", latencies)
print("mean:", round(statistics.mean(latencies), 3), "s")
print("p50 :", round(statistics.median(latencies), 3), "s")
print("max :", round(max(latencies), 3), "s")


In [ ]:
# 5) JSON 输出能力测试（常用于 agent/router）
schema_prompt = "只输出 JSON：{\"action_kind\":\"observe|generate_task\",\"reason\":\"...\"}。用户请求：帮我做一轮回归测试。"
out = chat_once(schema_prompt, temperature=0.0, max_tokens=120)
print(out["text"])
print("latency:", out["latency_s"], "s")


## 说明

- 若连接失败，先确认 sglang 进程是否在运行。
- 可通过环境变量覆盖：SGLANG_BASE_URL、SGLANG_API_KEY、SGLANG_MODEL。
- 若要压测吞吐，可把第 4 节改成并发调用（线程池或异步）。
